<a href="https://colab.research.google.com/github/shaikgaleebshaheed/Mobile-Usage-and-Productivity-Analysis/blob/main/DWDM_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import joblib

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Smartphone Usage Productivity Analysis.csv to Smartphone Usage Productivity Analysis.csv


In [ ]:
import pandas as pd

file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset shape: (8000, 16)


,transaction_id,user_id,age,gender,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,stress_level,academic_work_impact,addiction_level,addicted_label
0,1,10715,19.0,Male,2.53,3.91,0.45,5.52,8.93,73.0,62.0,4.04,Moderate,Moderate impact,4.0,No
1,2,16192,15.0,Female,5.83,3.06,0.11,3.24,5.89,67.0,69.0,6.19,Moderate,Slight impact,5.0,No
2,3,15237,38.0,Female,4.83,1.66,0.40,6.91,5.08,59.0,67.0,8.19,Moderate,Slight impact,1.0,No
3,4,13512,46.0,Female,1.25,2.21,1.57,2.21,4.79,67.0,93.0,8.82,Low,Moderate impact,3.0,No
4,5,13465,23.0,Female,6.52,4.48,0.46,2.87,8.05,67.0,81.0,3.23,Low,High impact,7.0,No


In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values in each column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Column names:
['transaction_id', 'user_id', 'age', 'gender', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'stress_level', 'academic_work_impact', 'addiction_level', 'addicted_label']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   transaction_id           8000 non-null   int64  
 1   user_id                  8000 non-null   int64  
 2   age                      7980 non-null   float64
 3   gender                   7992 non-null   object 
 4   daily_screen_time_hours  7982 non-null   float64
 5   social_media_hours       7985 non-null   float64
 6   gaming_hours             7988 non-null   float64
 7   work_study_hours         7988 non-null   float64
 8   sleep_hours        

In [ ]:
required_columns = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day",
    "addicted_label"
]

df = df[required_columns].copy()

print("Required columns selected successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Required columns selected successfully!
Dataset shape: (8000, 8)


,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,addicted_label
0,19.0,2.53,3.91,0.45,5.52,8.93,73.0,No
1,15.0,5.83,3.06,0.11,3.24,5.89,67.0,No
2,38.0,4.83,1.66,0.40,6.91,5.08,59.0,No
3,46.0,1.25,2.21,1.57,2.21,4.79,67.0,No
4,23.0,6.52,4.48,0.46,2.87,8.05,67.0,No


In [ ]:
numeric_columns = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

df = df.dropna(subset=["addicted_label"])
df["addicted_label"] = df["addicted_label"].astype(str).str.strip()

df = df[df["addicted_label"].isin(["Yes", "No"])]
df = df.drop_duplicates()
df = df.reset_index(drop=True)

print("Data cleaning completed!")
print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

Data cleaning completed!
Dataset shape: (7985, 8)

Missing values:
age                        0
daily_screen_time_hours    0
social_media_hours         0
gaming_hours               0
work_study_hours           0
sleep_hours                0
notifications_per_day      0
addicted_label             0
dtype: int64


In [ ]:
high_risk = (
    (df["daily_screen_time_hours"] >= 8) &
    (
        (df["social_media_hours"] + df["gaming_hours"] >= 4) |
        (df["sleep_hours"] < 6)
    )
)

df["addicted_label"] = "No"
df.loc[high_risk, "addicted_label"] = "Yes"

print("Target labels updated successfully!")
print("\nLabel counts:")
print(df["addicted_label"].value_counts())

print("\nDataset shape:", df.shape)


Target labels updated successfully!

Label counts:
addicted_label
No     7671
Yes     314
Name: count, dtype: int64

Dataset shape: (7985, 8)


In [ ]:
print("Final dataset shape:", df.shape)

print("\nTarget label distribution:")
print(df["addicted_label"].value_counts())

print("\nTarget label percentages:")
print((df["addicted_label"].value_counts(normalize=True) * 100).round(2))

Final dataset shape: (7985, 8)

Target label distribution:
addicted_label
No     7671
Yes     314
Name: count, dtype: int64

Target label percentages:
addicted_label
No     96.07
Yes     3.93
Name: proportion, dtype: float64


In [ ]:
print("Final dataset shape:", df.shape)

print("\nTarget label distribution:")
print(df["addicted_label"].value_counts())

print("\nTarget label percentages:")
print((df["addicted_label"].value_counts(normalize=True) * 100).round(2))

Final dataset shape: (7985, 8)

Target label distribution:
addicted_label
No     7671
Yes     314
Name: count, dtype: int64

Target label percentages:
addicted_label
No     96.07
Yes     3.93
Name: proportion, dtype: float64


In [ ]:
features = [
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "sleep_hours",
    "work_study_hours"
]

X = df[features]
y = LabelEncoder().fit_transform(df["addicted_label"])

print("Input features selected:", features)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target encoding: No = 0, Yes = 1")

Input features selected: ['daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'sleep_hours', 'work_study_hours']
X shape: (7985, 5)
y shape: (7985,)
Target encoding: No = 0, Yes = 1


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

Training data: (6388, 5)
Testing data: (1597, 5)
Training labels: (6388,)
Testing labels: (1597,)


In [ ]:
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed!")
print("Scaled training data:", X_train_scaled.shape)
print("Scaled testing data:", X_test_scaled.shape)

Feature scaling completed!
Scaled training data: (6388, 5)
Scaled testing data: (1597, 5)


In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),

    "Naive Bayes": GaussianNB(),

    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),

    "SVM": SVC(
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42
    )
}

for name, model in models.items():
    if name in ["Logistic Regression", "SVM"]:
        model.fit(X_train_scaled, y_train)
    else:
        model.fit(X_train, y_train)

print("All five models trained successfully!")

All five models trained successfully!


In [ ]:
results = []

for name, model in models.items():

    if name in ["Logistic Regression", "SVM"]:
        predictions = model.predict(X_test_scaled)
    else:
        predictions = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1 Score": f1_score(y_test, predictions, zero_division=0)
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)

display(results_df.round(4))

,Model,Accuracy,Precision,Recall,F1 Score
0,Random Forest,0.9962,0.9831,0.9206,0.9508
1,Decision Tree,0.9956,0.9667,0.9206,0.9431
2,SVM,0.9505,0.4437,1.0000,0.6146
3,Naive Bayes,0.9756,1.0000,0.3810,0.5517
4,Logistic Regression,0.9036,0.2903,1.0000,0.4500


In [ ]:
best_model_name = results_df.loc[0, "Model"]
best_model = models[best_model_name]

print("Best Model:", best_model_name)
print("\nBest Model Performance:")
print(results_df.iloc[0])

Best Model: Random Forest

Best Model Performance:
Model        Random Forest
Accuracy          0.996243
Precision         0.983051
Recall            0.920635
F1 Score           0.95082
Name: 0, dtype: object


In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(["No", "Yes"])

joblib.dump(best_model, "best_model.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")

print("Best model:", best_model_name)
print("Model saved as best_model.pkl")
print("Label encoder saved as label_encoder.pkl")

Best model: Random Forest
Model saved as best_model.pkl
Label encoder saved as label_encoder.pkl


In [ ]:
from google.colab import files

files.download("best_model.pkl")
files.download("label_encoder.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>